## Demo: Introduction to Spark Structured Streaming



databricks https://dbc-0b69ef93-5247.cloud.databricks.com/editor/notebooks/2823592720212150?o=7474657872577658

Curso: https://customer-academy.databricks.com/learn/courses/3959/stream-processing-and-analysis-with-apache-spark/lessons/36561/demo-introduction-to-spark-structured-streaming?hash=3e063f193bec2e8ca7cdf56ffbdced1657bcc087&generated_by=409740

Stream Processing and Analysis with Apache Spark | 2. Spark Structured Streaming

# 2.2 Demo - Introduction to Spark Structured Streaming

**Duration:** ~25 minutes

This demo applies the Structured Streaming API to the Bakehouse orders file stream created during Required Setup. You will create a streaming DataFrame over arriving JSON files, transform it, write it to memory and Delta sinks, combine streams with `union`, control processing with triggers, and monitor query progress.

## Learning Objectives

By the end of this demo, you will be able to

## B. Stream Processing Setup

First, let's set up our streaming infrastructure: resolve the volume paths, define the schema for the order events, and create the streaming DataFrame. Streaming file sources require an explicit schema, and `maxFilesPerTrigger` throttles the stream to one file per micro-batch so we can watch it process incrementally.

In [0]:
# Create the Orders Streaming DataFrame
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import json

# Obtém o catálogo atual e define os caminhos do Volume e dos checkpoints
catalog = spark.sql("SELECT current_catalog()").first()[0]
volume_root = f"/Volumes/{catalog}/spark_stream/stream_source"
checkpoint_root = f"/Volumes/{catalog}/spark_stream/stream_checkpoints"
print(catalog)

In [0]:
import json

# Caminhos /Volumes/... são volumes do Unity Catalog, não pastas locais:
# os.makedirs/open não funcionam. Cria-se o schema/volumes no catálogo e grava-se com dbutils.fs.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.spark_stream")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalog}`.spark_stream.stream_source")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalog}`.spark_stream.stream_checkpoints")

dbutils.fs.mkdirs(f"{volume_root}/orders")
dbutils.fs.mkdirs(checkpoint_root)

# Registros compatíveis com orders_schema da seção B
batch_1 = [
    {
        "order_id": 101,
        "customer_id": 5001,
        "franchise_id": 10,
        "order_timestamp": 1795280400,
        "product": "Croissant",
        "quantity": 12,
        "total_price": 42
    },
    {
        "order_id": 102,
        "customer_id": 5002,
        "franchise_id": 10,
        "order_timestamp": 1795280460,
        "product": "Sourdough Loaf",
        "quantity": 30,
        "total_price": 180
    }
]

batch_2 = [
    {
        "order_id": 103,
        "customer_id": 5003,
        "franchise_id": 12,
        "order_timestamp": 1795280520,
        "product": "Espresso",
        "quantity": 24,
        "total_price": 72
    }
]

# Grava os arquivos JSON (um registro por linha — formato padrão JSON Lines do Spark)
# na pasta orders/, que é exatamente a fonte lida pelo readStream
dbutils.fs.put(f"{volume_root}/orders/orders_01.json", "\n".join(json.dumps(row) for row in batch_1) + "\n", overwrite=True)
dbutils.fs.put(f"{volume_root}/orders/orders_02.json", "\n".join(json.dumps(row) for row in batch_2) + "\n", overwrite=True)

print(f"Arquivos JSON criados em: {volume_root}/orders")

Wrote 304 bytes.
Wrote 148 bytes.
Arquivos JSON criados em: /Volumes/workspace/spark_stream/stream_source/orders


In [0]:
### Célula de Código: Create the Orders Streaming DataFrame

#python
# Create the Orders Streaming DataFrame
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Obtém o catálogo atual e define os caminhos do Volume e dos checkpoints
catalog = spark.sql("SELECT current_catalog()").first()[0]
volume_root = f"/Volumes/{catalog}/spark_stream/stream_source"
checkpoint_root = f"/Volumes/{catalog}/spark_stream/stream_checkpoints"
print(catalog)

workspace


In [0]:


# Esquema explícito obrigatório para fontes de ficheiros em streaming
orders_schema = StructType([
    StructField("order_id", LongType(), True),
    StructField("customer_id", LongType(), True),
    StructField("franchise_id", LongType(), True),
    StructField("order_timestamp", LongType(), True),
    StructField("product", StringType(), True),
    StructField("quantity", LongType(), True),
    StructField("total_price", LongType(), True),
])

# Cria o streaming DataFrame a partir dos ficheiros JSON (1 ficheiro por micro-batch)
orders_stream = (
    spark.readStream
    .format("json")
    .schema(orders_schema)
    .option("maxFilesPerTrigger", 1)
    .load(f"{volume_root}/orders")
)




=> **Nota para execução local / fora da aula:** Se ainda não tiveres os diretórios `/Volumes/{catalog}/spark_stream/...` criados ou ficheiros JSON dentro de `{volume_root}/orders`, podes criar as pastas com `dbutils.fs.mkdirs(...)` e gravar um ficheiro JSON de exemplo antes de iniciar o `writeStream`.

In [0]:
# Confirm We Have a Streaming DataFrame
print(f"isStreaming: {orders_stream.isStreaming}")

isStreaming: True


## C. Running a Basic Streaming Query

Calling `display()` on a streaming DataFrame starts an implicit streaming query and renders live results - the notebook equivalent of the console sink. Watch the row count climb as each micro-batch picks up another file, then stop the cell.

In [0]:
# Display the Stream
# Checkpoint explícito é obrigatório neste workspace; subdiretório único por execução
import time
display(orders_stream, checkpointLocation=f"{checkpoint_root}/orders_display/{int(time.time())}")

Checkpointing to /Volumes/workspace/spark_stream/stream_checkpoints/orders_display/1795293459


## D. Basic Transformations on the Stream

Stateless streaming transformations are exactly the narrow transformations you would perform on a static DataFrame: `select`, `filter`, `withColumn`, and friends. Each record is processed independently - no state is carried between micro-batch

In [0]:
# Transform the Stream
transformed_stream = (
    orders_stream
    .withColumn("order_time", to_timestamp(from_unixtime(col("order_timestamp"))))
    .withColumn("order_details", concat(lit("Order #"), col("order_id").cast("string")))
    .withColumn("is_bulk_order", col("quantity") >= 24)
)

In [0]:
import time
display(transformed_stream, checkpointLocation=f"{checkpoint_root}/transformed_display/{int(time.time())}")

Checkpointing to /Volumes/workspace/spark_stream/stream_checkpoints/transformed_display/1795293551


In [0]:
# Filter the Stream to Bulk Orders Only
bulk_orders_stream = orders_stream.filter(col("quantity") >= 24)

#display(bulk_orders_stream)

In [0]:
display(bulk_orders_stream, checkpointLocation=f"{checkpoint_root}/bulk_orders_stream/{int(time.time())}")

Checkpointing to /Volumes/workspace/spark_stream/stream_checkpoints/bulk_orders_stream/1795294065



## E. Writing to Sinks



Various streaming sinks are available, including:

- **Delta sink:** writes output to Delta tables - the standard production sink on Databricks
- **File sink:** writes output data to files in a directory
- **Kafka sink:** sends output records to Kafka topics
- **Memory sink:** stores output in memory for interactive querying (testing and debugging)
- **Console sink:** displays output in the console (debugging)
- **Foreach sink:** runs arbitrary computation on output records

When defining a streaming sink, you typically specify:

1. The output format with `.format()`
2. The output mode (`append`, `update`, or `complete`)
3. Any sink-specific options
4. A checkpoint location for fault tolerance

### Memory Sink

The memory sink materializes the stream into an in-memory temp view named by `queryName()`, which we can then query with SQL. The `availableNow` trigger processes everything currently in the source (one file per batch, because of `maxFilesPerTrigger`) and then stops; `awaitTermination()` blocks until it finishes.

In [0]:
# Write the Stream to a Memory Sink
import time

for q in spark.streams.active:
    if q.name == "orders_streaming_table":
        q.stop()

memory_query = (
    orders_stream.writeStream
    .format("memory")
    .queryName("orders_streaming_table")
    .outputMode("append")
    .option("checkpointLocation", f"{checkpoint_root}/memory_sink/{int(time.time())}")
    .trigger(availableNow=True)
    .start()
)

memory_query.awaitTermination()


### Query the In-Memory Table with SQL

In [0]:
%sql
-- Query the In-Memory Table with SQL
SELECT product, COUNT(*) AS num_orders, SUM(total_price) AS revenue
FROM orders_streaming_table
GROUP BY product


product,num_orders,revenue
Croissant,1,42
Sourdough Loaf,1,180
Espresso,1,72


### Delta Sink

The Delta sink is the production pattern: append the stream to a Delta table with a checkpoint location so the query can recover exactly where it left off. `toTable()` both starts the query and registers the target table.

In [0]:
# Write the Stream to a Delta Table
bronze_query = (
    orders_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{checkpoint_root}/orders_bronze")
    .trigger(availableNow=True)
    .toTable("orders_bronze")
)


In [0]:
bronze_query.awaitTermination()

### Verify the Delta Table

In [0]:
%sql
-- Verify the Delta Table
SELECT COUNT(*) AS order_count FROM orders_bronze;


order_count
3



Because the checkpoint recorded which files were processed, re-running the previous Python cell processes nothing new - the query starts, finds no unprocessed files, and stops. That is incremental processing with exactly-once delivery.


## F. Combining Multiple Streams Using Union

Different stream sources can be combined using relational operators like `union`. Here we split the orders stream into two filtered streams and recombine them.

> NOTE: Stream-to-static DataFrame joins are fully supported and easy to implement (for example, joining a stream with reference data for enrichment). Stream-to-stream joins are supported as well, but require special handling for state management. Both are covered in section 3.

In [0]:
# Union Two Filtered Streams
bulk_stream = orders_stream.filter(col("quantity") >= 24)
regular_stream = orders_stream.filter(col("quantity") < 24)

combined_stream = bulk_stream.union(regular_stream)

import time
display(combined_stream, checkpointLocation=f"{checkpoint_root}/combined_display/{int(time.time())}")

Checkpointing to /Volumes/workspace/spark_stream/stream_checkpoints/combined_display/1795295062


## G. Using Triggers to Control Processing

Triggers control when micro-batches run. Here we stamp each record with the time its micro-batch was processed; grouping by that timestamp afterwards shows exactly how the stream was carved into batches - one per file, thanks to `maxFilesPerTrigger`.

In [0]:
# Run a Triggered Query That Stamps Each Micro-Batch
import time

for q in spark.streams.active:
    if q.name == "triggered_query_table":
        q.stop()

triggered_query = (
    orders_stream
    .withColumn("processing_ts", current_timestamp())
    .writeStream
    .format("memory")
    .queryName("triggered_query_table")
    .outputMode("append")
    .option("checkpointLocation", f"{checkpoint_root}/triggered_query/{int(time.time())}")
    .trigger(availableNow=True)
    # .trigger(processingTime="1 second")
    .start()
)


In [0]:
# Let it run for 5 seconds
triggered_query.awaitTermination(5)

# Then explicitly stop it
triggered_query.stop()


### Inspect the Micro-Batches


In [0]:
%sql
-- Inspect the Micro-Batches
SELECT processing_ts, COUNT(*) AS record_count
FROM triggered_query_table
GROUP BY processing_ts
ORDER BY processing_ts;


processing_ts,record_count
2026-11-21T21:09:10.081Z,2
2026-11-21T21:09:10.957Z,1


In [0]:
%sql
-- Inspect the Micro-Batches
SELECT *
FROM triggered_query_table



order_id,customer_id,franchise_id,order_timestamp,product,quantity,total_price,processing_ts
101,5001,10,1795280400,Croissant,12,42,2026-11-21T21:09:10.081Z
102,5002,10,1795280460,Sourdough Loaf,30,180,2026-11-21T21:09:10.081Z
103,5003,12,1795280520,Espresso,24,72,2026-11-21T21:09:10.957Z


In [0]:
%sql
-- Inspect the Micro-Batches
SELECT *,
date_format(processing_ts,'yyyyMM') AS anomes
-- date_format(processing_ts,'%Ÿ%m') AS anomes

FROM triggered_query_table;


order_id,customer_id,franchise_id,order_timestamp,product,quantity,total_price,processing_ts,anomes
101,5001,10,1795280400,Croissant,12,42,2026-11-21T21:09:10.081Z,202611
102,5002,10,1795280460,Sourdough Loaf,30,180,2026-11-21T21:09:10.081Z,202611
103,5003,12,1795280520,Espresso,24,72,2026-11-21T21:09:10.957Z,202611


## H. Monitoring Streaming Queries

Every streaming query exposes its identity, current status, and most recent progress report. `lastProgress` is the workhorse: rows per second, batch durations, and source offsets. The same information is visualized in the Structured Streaming tab of the Spark UI.

### Inspect Query Status and Progress


In [0]:
# Inspect Query Status and Progress
import json

print(f"Query id : {triggered_query.id}")
print(f"Status   : {triggered_query.status}")
print(json.dumps(triggered_query.lastProgress, indent=2, default=str))

Query id : ab7e5873-dbf7-4800-a1a4-fcd46499632a
Status   : {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}
{
  "id": "ab7e5873-dbf7-4800-a1a4-fcd46499632a",
  "runId": "eb528223-f1d2-4815-8f0b-1466de72f766",
  "name": "triggered_query_table",
  "timestamp": "2026-11-21T21:09:10.882Z",
  "batchId": 1,
  "batchDuration": 587,
  "durationMs": {
    "triggerExecution": 587,
    "queryPlanning": 27,
    "collectSourceMetrics": 0,
    "getBatch": 188,
    "commitOffsets": 91,
    "addBatch": 114,
    "latestOffset": 75,
    "walCommit": 90
  },
  "eventTime": {},
  "stateOperators": [],
  "sources": [
    {
      "description": "FileStreamSource[dbfs:/Volumes/workspace/spark_stream/stream_source/orders]",
      "startOffset": "{\"logOffset\":0}",
      "endOffset": "{\"logOffset\":1}",
      "latestOffset": "null",
      "numInputRows": 1,
      "inputRowsPerSecond": 1.1299435028248588,
      "processedRowsPerSecond": 1.7035775127768313,
      "metrics": {}
    }
 

### Stop All Active Streaming Queries

Run the cell below to stop any remaining active streaming queries.

In [0]:

# Stop All Active Streaming Queries
for query in spark.streams.active:
    query.stop()

print(f"Active streaming queries: {len(spark.streams.active)}")

Active streaming queries: 0


## Summary



In this demo you:

- Created a streaming DataFrame over the orders JSON files with an explicit schema and `maxFilesPerTrigger` throttling
- Applied stateless transformations (`withColumn`, `filter`) just as you would on a static DataFrame
- Wrote the stream to a memory sink for interactive SQL and to a Delta table with a checkpoint for exactly-once incremental processing
- Combined filtered streams with `union`
- Used the `availableNow` trigger to process all available data and stop, and inspected the per-file micro-batches it produced
- Monitored queries through `status` and `lastProgress`, and stopped all active queries cleanly


Concluíste toda a demonstração **2.2 Demo - Introduction to Spark Structured Streaming**! Queres avançar agora para o **2.3 Lab – Building Streaming Queries**?